# Gradients, Hessians and Optima

At an interior optimum of $f(x_1, \dots, x_n)$ the gradient is zero (first-order condition). The Hessian's eigenvalues then say whether the point is a maximum, a minimum or a saddle (second-order condition). This notebook uses a profit function of two inputs.

In [ ]:
Pollis.packages("DifferentiationInterface", "Plots", "ForwardDiff")

In [ ]:
using DifferentiationInterface, LinearAlgebra, Plots
import ForwardDiff
backend = AutoForwardDiff()

## Profit of a firm with a Cobb-Douglas technology

Output $q = K^{0.3} L^{0.5}$ sells at price $p = 10$; capital costs $r = 3.4$ and labour $w = 2.5$. The optimum is near $K = 4$, $L = 9$. Decreasing returns ($0.3 + 0.5 < 1$) make profit concave.

In [ ]:
p, r, w = 10.0, 3.4, 2.5
profit(x) = p * x[1]^0.3 * x[2]^0.5 - r * x[1] - w * x[2]
x0 = [3.0, 8.0]
(gradient = gradient(profit, backend, x0), hessian = hessian(profit, backend, x0))

## Solve the first-order conditions with Newton's method

Newton's method uses exactly these two objects: $x_{k+1} = x_k - H(x_k)^{-1} \nabla f(x_k)$.

In [ ]:
x = copy(x0)
for k in 1:20
    g = gradient(profit, backend, x)
    norm(g) < 1e-10 && break
    x -= hessian(profit, backend, x) \ g
    println(k, "  x = ", round.(x, digits=6), "  |gradient| = ", norm(g))
end
x

## Classify the stationary point

In [ ]:
H = hessian(profit, backend, x)
ev = eigvals(Symmetric(H))
(stationary_point = x, profit = profit(x), eigenvalues = ev, kind = all(<(0), ev) ? "maximum" : all(>(0), ev) ? "minimum" : "saddle")

In [ ]:
ks = range(0.5, 15, length=150); ls = range(0.5, 20, length=150)
contour(ks, ls, (k, l) -> profit([k, l]), levels=30, xlabel="K", ylabel="L", title="Profit")
scatter!([x[1]], [x[2]], c=:red, ms=6, label="optimum")

## Exercises

1. Set the exponents to $0.5$ and $0.5$ (constant returns). What happens to the Hessian and to Newton's method?
2. Replace the manual loop with the Local Optimisation panel's `Optim.optimize` and compare.